# Stage 3 – Variant Pathogenicity Analysis

Interactive exploration of SCN1A/SCN2A missense variant predictions from three orthogonal tools:
- **AlphaMissense** — structure-based ML predictor (Cheng et al. 2023)
- **ESM-2/ESM-1b** — evolutionary language model (Meier et al. 2021; uses ESM-2)
- **PRESCOTT** — evolutionary conservation metric

## Sections
1. Load predictions
2. Score distributions
3. Variant pathogenicity heatmap
4. Consensus scoring
5. Clinical validation (ClinVar)

**Prerequisites**:
```
python stage3_variant_pathogenicity/01_generate_variants.py
python stage3_variant_pathogenicity/02_run_alphamissense.py
python stage3_variant_pathogenicity/03_run_esm1b.py
python stage3_variant_pathogenicity/04_run_prescott.py
python stage3_variant_pathogenicity/05_consensus_analysis.py
python stage3_variant_pathogenicity/06_compare_clinical_data.py
```

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../..'))

RESULTS_DIR = '../../data/results/stage3'
GENE = 'SCN1A'  # Change to 'SCN2A' as needed
print('Imports OK')

## 1. Load Predictions

In [ ]:
consensus_path = os.path.join(RESULTS_DIR, f'{GENE.lower()}_consensus.csv')

if os.path.exists(consensus_path):
    df = pd.read_csv(consensus_path)
    print(f'Loaded {len(df)} variant predictions for {GENE}')
    display(df.head())
else:
    print(f'Consensus CSV not found — generating mock data for {GENE}')
    rng = np.random.default_rng(42)
    n = 2000
    AA = list('ACDEFGHIKLMNPQRSTVWY')
    pos = rng.integers(1, 2010, n)
    wt = rng.choice(list('ACDEFGHIKLMNPQRSTVWY'), n)
    mut = rng.choice(list('ACDEFGHIKLMNPQRSTVWY'), n)
    am_score = rng.beta(2, 2, n)
    esm_llr = rng.normal(-1.5, 3, n)
    prescott_score = rng.beta(1.5, 2, n)
    consensus = 0.45*am_score + 0.35*np.clip(-esm_llr/20, 0, 1) + 0.20*prescott_score
    df = pd.DataFrame({
        'gene': GENE, 'position': pos, 'wt_aa': wt, 'mut_aa': mut,
        'alphamissense_score': am_score,
        'esm_llr': esm_llr,
        'prescott_score': prescott_score,
        'consensus_score': consensus,
        'consensus_class': np.where(consensus >= 0.564, 'pathogenic',
                           np.where(consensus <= 0.34, 'benign', 'uncertain')),
    })
    print(f'Mock data: {len(df)} variants')
    display(df.head())

## 2. Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, label, color) in zip(axes, [
    ('alphamissense_score', 'AlphaMissense', '#1565C0'),
    ('esm_llr', 'ESM-2 LLR', '#E53935'),
    ('consensus_score', 'Consensus', '#43A047'),
]):
    if col in df.columns:
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.75, density=True)
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
        ax.set_title(f'{label} Distribution')
        ax.grid(alpha=0.3)

plt.suptitle(f'{GENE} — Score Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Pathogenicity Heatmap

Position × mutant amino acid heatmap of consensus pathogenicity scores.

In [ ]:
AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')

# Subset to first 100 positions for display performance
pos_subset = sorted(df['position'].unique())[:100]
df_sub = df[df['position'].isin(pos_subset)]

matrix = np.full((len(pos_subset), len(AA_ORDER)), np.nan)
pos_idx = {p: i for i, p in enumerate(pos_subset)}
aa_idx = {aa: i for i, aa in enumerate(AA_ORDER)}

for _, row in df_sub.iterrows():
    pi = pos_idx.get(row['position'])
    ai = aa_idx.get(row['mut_aa'])
    if pi is not None and ai is not None:
        matrix[pi, ai] = row['consensus_score']

fig, ax = plt.subplots(figsize=(18, 5))
im = ax.imshow(matrix.T, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=1, interpolation='nearest')
plt.colorbar(im, ax=ax, label='Consensus Pathogenicity', shrink=0.8)
ax.set_xticks(np.arange(len(pos_subset)))
ax.set_xticklabels(pos_subset, rotation=90, fontsize=5)
ax.set_yticks(np.arange(len(AA_ORDER)))
ax.set_yticklabels(AA_ORDER, fontsize=8)
ax.set_xlabel('Residue Position')
ax.set_ylabel('Mutant Amino Acid')
ax.set_title(f'{GENE} — Variant Pathogenicity Heatmap (first 100 positions)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Clinical Validation

Compare predictions against ClinVar-annotated variants.

In [ ]:
clinvar_path = os.path.join(RESULTS_DIR, f'{GENE.lower()}_clinical_comparison.csv')

if os.path.exists(clinvar_path):
    df_cv = pd.read_csv(clinvar_path)
    print(f'ClinVar comparison: {len(df_cv)} annotated variants')
    # Classification counts
    display(df_cv.groupby(['clinical_significance', 'consensus_class']).size().unstack(fill_value=0))
else:
    print('Clinical comparison CSV not found — run stage3/06_compare_clinical_data.py')
    # Show class counts from consensus
    if 'consensus_class' in df.columns:
        counts = df['consensus_class'].value_counts()
        fig, ax = plt.subplots(figsize=(6, 4))
        colors = {'pathogenic': '#E53935', 'benign': '#43A047', 'uncertain': '#FFB300'}
        bars = ax.bar(counts.index, counts.values,
                     color=[colors.get(c, '#999') for c in counts.index])
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                   ha='center', fontsize=10)
        ax.set_title(f'{GENE} Consensus Classification')
        ax.set_ylabel('Count')
        plt.tight_layout()
        plt.show()